# QuantumNet - Kaggle Version (CNN, ResNet50, MobileNetV2 baselines)
Run cells top to bottom. Before running: right panel > Settings > Accelerator = GPU, and Add Input = APTOS 2019 Blindness Detection.

## Step 1: GPU check

In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Step 2: Find dataset files automatically

In [2]:
import os, glob
csv_files = glob.glob('/kaggle/input/**/train.csv', recursive=True)
print(csv_files)
CSV_PATH = csv_files[0]
BASE = os.path.dirname(CSV_PATH)
IMG_DIR = os.path.join(BASE, 'train_images')
print(CSV_PATH); print(IMG_DIR, os.path.exists(IMG_DIR))

['/kaggle/input/competitions/aptos2019-blindness-detection/train.csv']
/kaggle/input/competitions/aptos2019-blindness-detection/train.csv
/kaggle/input/competitions/aptos2019-blindness-detection/train_images True


## Step 3: Binary labels + 70/15/15 split (seed 42)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
SEED = 42
df = pd.read_csv(CSV_PATH)
print(df['diagnosis'].value_counts().sort_index())
df['label'] = (df['diagnosis'] > 0).astype(int)
print(df['label'].value_counts())
ext = os.listdir(IMG_DIR)[0].split('.')[-1]
df['path'] = IMG_DIR + '/' + df['id_code'] + '.' + ext
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))   # expect 2563 549 550
os.makedirs('/kaggle/working', exist_ok=True)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(f'/kaggle/working/{name}_split.csv', index=False)

diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64
label
1    1857
0    1805
Name: count, dtype: int64
2563 549 550


## Step 4: Dataset + augmentation

In [4]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
torch.manual_seed(SEED); np.random.seed(SEED)
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomAffine(0, translate=(0.1, 0.1), shear=10, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)])

class DRDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame['path'].tolist(); self.labels = frame['label'].tolist(); self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.labels[i]

BS = 32
train_loader = DataLoader(DRDataset(train_df, train_tf), batch_size=BS, shuffle=True, num_workers=2)
val_loader   = DataLoader(DRDataset(val_df, eval_tf), batch_size=BS, num_workers=2)
test_loader  = DataLoader(DRDataset(test_df, eval_tf), batch_size=BS, num_workers=2)
x, y = next(iter(train_loader)); print(x.shape, y[:8])

torch.Size([32, 3, 224, 224]) tensor([1, 0, 1, 1, 1, 1, 0, 0])


## Step 5: Reusable train + test functions

In [5]:
import torch.nn as nn, torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
criterion = nn.CrossEntropyLoss()
results = []

def run_epoch(net, loader, train, opt=None):
    net.train() if train else net.eval()
    tot_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = net(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot_loss / n, correct / n

def train_and_test(net, params, name, epochs=8, lr=1e-3):
    net = net.to(device)
    opt = torch.optim.Adam(params, lr=lr)
    best, path = 0, f'/kaggle/working/{name}_best.pt'
    for ep in range(epochs):
        tl, ta = run_epoch(net, train_loader, True, opt)
        vl, va = run_epoch(net, val_loader, False)
        print(f'{name} epoch {ep+1}: train acc {ta:.3f} | val loss {vl:.3f} val acc {va:.3f}')
        if va > best:
            best = va; torch.save(net.state_dict(), path)
    net.load_state_dict(torch.load(path)); net.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in test_loader:
            ps += net(x.to(device)).argmax(1).cpu().tolist(); ys += y.tolist()
    print(classification_report(ys, ps, target_names=['No DR', 'DR'], digits=4))
    print(confusion_matrix(ys, ps))
    results.append((name, round(accuracy_score(ys, ps), 4)))
    return net

## Step 6: ResNet50 (frozen backbone, new 2-class head)

In [6]:
resnet = models.resnet50(weights='IMAGENET1K_V2')
for p in resnet.parameters(): p.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet = train_and_test(resnet, resnet.fc.parameters(), 'resnet50')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]


resnet50 epoch 1: train acc 0.872 | val loss 0.315 val acc 0.916
resnet50 epoch 2: train acc 0.933 | val loss 0.260 val acc 0.931
resnet50 epoch 3: train acc 0.945 | val loss 0.212 val acc 0.938
resnet50 epoch 4: train acc 0.950 | val loss 0.226 val acc 0.923
resnet50 epoch 5: train acc 0.946 | val loss 0.176 val acc 0.953
resnet50 epoch 6: train acc 0.953 | val loss 0.218 val acc 0.920
resnet50 epoch 7: train acc 0.957 | val loss 0.196 val acc 0.938
resnet50 epoch 8: train acc 0.957 | val loss 0.181 val acc 0.938
              precision    recall  f1-score   support

       No DR     0.9377    0.9446    0.9412       271
          DR     0.9458    0.9391    0.9424       279

    accuracy                         0.9418       550
   macro avg     0.9418    0.9419    0.9418       550
weighted avg     0.9418    0.9418    0.9418       550

[[256  15]
 [ 17 262]]


## Step 7: Simple CNN

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))
    def forward(self, x): return self.classifier(self.features(x))

cnn = SimpleCNN()
cnn = train_and_test(cnn, cnn.parameters(), 'simple_cnn')

simple_cnn epoch 1: train acc 0.697 | val loss 0.346 val acc 0.843
simple_cnn epoch 2: train acc 0.805 | val loss 0.246 val acc 0.885
simple_cnn epoch 3: train acc 0.837 | val loss 0.350 val acc 0.882
simple_cnn epoch 4: train acc 0.863 | val loss 0.229 val acc 0.907
simple_cnn epoch 5: train acc 0.879 | val loss 0.212 val acc 0.913
simple_cnn epoch 6: train acc 0.885 | val loss 0.208 val acc 0.922
simple_cnn epoch 7: train acc 0.908 | val loss 0.227 val acc 0.933
simple_cnn epoch 8: train acc 0.909 | val loss 0.203 val acc 0.929
              precision    recall  f1-score   support

       No DR     0.9112    0.8708    0.8906       271
          DR     0.8797    0.9176    0.8982       279

    accuracy                         0.8945       550
   macro avg     0.8955    0.8942    0.8944       550
weighted avg     0.8952    0.8945    0.8945       550

[[236  35]
 [ 23 256]]


## Step 8: MobileNetV2 (frozen backbone)

In [8]:
mobilenet = models.mobilenet_v2(weights='IMAGENET1K_V1')
for p in mobilenet.parameters(): p.requires_grad = False
mobilenet.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(mobilenet.last_channel, 2))
mobilenet = train_and_test(mobilenet, mobilenet.classifier.parameters(), 'mobilenetv2')

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 63.1MB/s]


mobilenetv2 epoch 1: train acc 0.873 | val loss 0.150 val acc 0.956
mobilenetv2 epoch 2: train acc 0.929 | val loss 0.137 val acc 0.960
mobilenetv2 epoch 3: train acc 0.942 | val loss 0.121 val acc 0.964
mobilenetv2 epoch 4: train acc 0.948 | val loss 0.149 val acc 0.951
mobilenetv2 epoch 5: train acc 0.935 | val loss 0.120 val acc 0.964
mobilenetv2 epoch 6: train acc 0.935 | val loss 0.099 val acc 0.969
mobilenetv2 epoch 7: train acc 0.941 | val loss 0.245 val acc 0.905
mobilenetv2 epoch 8: train acc 0.941 | val loss 0.104 val acc 0.958
              precision    recall  f1-score   support

       No DR     0.9481    0.9446    0.9464       271
          DR     0.9464    0.9498    0.9481       279

    accuracy                         0.9473       550
   macro avg     0.9473    0.9472    0.9473       550
weighted avg     0.9473    0.9473    0.9473       550

[[256  15]
 [ 14 265]]


## Step 9: Final results table

In [9]:
pd.DataFrame(results, columns=['Model', 'Test Accuracy'])

,Model,Test Accuracy
0,resnet50,0.9418
1,simple_cnn,0.8945
2,mobilenetv2,0.9473


## Save: click 'Save Version' (top right) > 'Save & Run All' so the .pt files and split CSVs in /kaggle/working are kept.

In [10]:
import torch.nn as nn, torchvision.models as models
from sklearn.metrics import accuracy_score

In [11]:
resnet_fe = models.resnet50(weights='IMAGENET1K_V2')
resnet_fe.fc = nn.Identity()          # last layer hata do -> 2048 features milenge
resnet_fe = resnet_fe.to(device).eval()

def extract(loader):
    feats, labs = [], []
    with torch.no_grad():
        for x, y in loader:
            feats.append(resnet_fe(x.to(device)).cpu().numpy())
            labs += y.tolist()
    return np.concatenate(feats), np.array(labs)

# train ke liye augmentation nahi lagana (eval_tf use kar rahe hain)
tr_loader_noaug = DataLoader(DRDataset(train_df, eval_tf), batch_size=BS, num_workers=2)
Xtr, ytr = extract(tr_loader_noaug)
Xva, yva = extract(val_loader)
Xte, yte = extract(test_loader)
print(Xtr.shape, Xva.shape, Xte.shape)

np.savez('/kaggle/working/resnet50_features.npz',
         Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte)

(2563, 2048) (549, 2048) (550, 2048)


In [12]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

pca = PCA(n_components=4, random_state=42).fit(Xtr)     # sirf train par fit
scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(Xtr))

def reduce(X):
    return np.clip(scaler.transform(pca.transform(X)), 0, np.pi)

Ztr, Zva, Zte = reduce(Xtr), reduce(Xva), reduce(Xte)
print('Variance kept by 4 components:', pca.explained_variance_ratio_.sum())

clf = LogisticRegression(max_iter=1000).fit(Ztr, ytr)
print('Classical check on 4 features, test acc:', accuracy_score(yte, clf.predict(Zte)))

np.savez('/kaggle/working/features_4d.npz',
         Ztr=Ztr, ytr=ytr, Zva=Zva, yva=yva, Zte=Zte, yte=yte)

Variance kept by 4 components: 0.43623826
Classical check on 4 features, test acc: 0.8181818181818182


In [13]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(Xtr)
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr_t, Xva_t, Xte_t = T(sc.transform(Xtr)), T(sc.transform(Xva)), T(sc.transform(Xte))
ytr_t, yva_t, yte_t = [torch.tensor(a, dtype=torch.long) for a in (ytr, yva, yte)]

torch.manual_seed(42)
pre  = nn.Sequential(nn.Linear(2048, 4), nn.Tanh())   # 4 numbers in (-1, 1)
head = nn.Linear(4, 2)
net  = nn.Sequential(pre, head)
opt  = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

best_va, best_state = 0, None
for ep in range(60):
    net.train()
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 64):
        idx = perm[i:i+64]
        opt.zero_grad(); loss = crit(net(Xtr_t[idx]), ytr_t[idx]); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        va = (net(Xva_t).argmax(1) == yva_t).float().mean().item()
    if va > best_va:
        best_va = va
        best_state = {k: v.clone() for k, v in net.state_dict().items()}

net.load_state_dict(best_state); net.eval()
with torch.no_grad():
    te = (net(Xte_t).argmax(1) == yte_t).float().mean().item()
print(f'Best val acc: {best_va:.4f} | Test acc (trainable 2048->4 bottleneck): {te:.4f}')

Best val acc: 0.9763 | Test acc (trainable 2048->4 bottleneck): 0.9527


In [14]:
!pip -q install pennylane
import pennylane as qml

n_qubits, n_layers = 4, 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(inputs, weights):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)                      # superposition
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)            # angle encoding
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.Rot(*weights[l, i], wires=i)       # trainable rotations
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])             # 3 CNOTs: entanglement
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

qlayer = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits, 3)})

class QuantumNetHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(2048, 4), nn.Tanh())
        self.q = qlayer
        self.out = nn.Linear(4, 2)
    def forward(self, x):
        z = self.pre(x) * np.pi                    # angles in (-pi, pi)
        return self.out(self.q(z))

# quick test
m = QuantumNetHead()
print(m(Xtr_t[:5]).shape)                          # expect torch.Size([5, 2])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 99.7 MB/s eta 0:00:00
torch.Size([5, 2])


In [15]:
import time
EPOCHS = 60
torch.manual_seed(42)
qnet = QuantumNetHead()
opt = torch.optim.Adam([
    {"params": qnet.pre.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    {"params": qnet.q.parameters(),   "lr": 1e-2},
    {"params": qnet.out.parameters(), "lr": 1e-2},
])
best_va, best_state = 0, None
t0 = time.time()
for ep in range(EPOCHS):
    qnet.train()
    perm = torch.randperm(len(Xtr_t))
    for i in range(0, len(perm), 64):
        idx = perm[i:i+64]
        opt.zero_grad(); loss = crit(qnet(Xtr_t[idx]), ytr_t[idx]); loss.backward(); opt.step()
    qnet.eval()
    with torch.no_grad():
        va = (qnet(Xva_t).argmax(1) == yva_t).float().mean().item()
    if va > best_va:
        best_va = va; best_state = {k: v.clone() for k, v in qnet.state_dict().items()}
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f'epoch {ep+1}: loss {loss.item():.3f} | val acc {va:.4f} | {time.time()-t0:.0f}s')

qnet.load_state_dict(best_state); qnet.eval()
with torch.no_grad():
    te = (qnet(Xte_t).argmax(1) == yte_t).float().mean().item()
print(f'QuantumNet head: best val {best_va:.4f} | test acc {te:.4f}')

epoch 1: loss 0.698 | val acc 0.9199 | 1s
epoch 5: loss 0.078 | val acc 0.9235 | 5s
epoch 10: loss 0.009 | val acc 0.9089 | 9s
epoch 15: loss 0.004 | val acc 0.9089 | 13s
epoch 20: loss 0.020 | val acc 0.8780 | 18s
epoch 25: loss 0.260 | val acc 0.7450 | 22s
epoch 30: loss 0.108 | val acc 0.8197 | 27s
epoch 35: loss 0.300 | val acc 0.8069 | 31s
epoch 40: loss 0.040 | val acc 0.8270 | 36s
epoch 45: loss 0.073 | val acc 0.8106 | 40s
epoch 50: loss 0.341 | val acc 0.7450 | 45s
epoch 55: loss 0.245 | val acc 0.7869 | 49s
epoch 60: loss 0.313 | val acc 0.7887 | 53s
QuantumNet head: best val 0.9271 | test acc 0.8927


In [16]:
class QuantumNetHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(2048, 4), nn.Tanh())
        self.q = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits, 3)})
        self.out = nn.Linear(4, 2)
    def forward(self, x):
        return self.out(self.q(self.pre(x) * np.pi))

def train_q(lr_pre, lr_q, wd, epochs=40, patience=8, seed=42):
    torch.manual_seed(seed)
    m = QuantumNetHead()
    opt = torch.optim.Adam([
        {"params": m.pre.parameters(), "lr": lr_pre, "weight_decay": wd},
        {"params": m.q.parameters(),   "lr": lr_q},
        {"params": m.out.parameters(), "lr": lr_q},
    ])
    best_va, best_state, bad = 0, None, 0
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]
            opt.zero_grad(); crit(m(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            va = (m(Xva_t).argmax(1) == yva_t).float().mean().item()
        if va > best_va:
            best_va, bad = va, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    m.load_state_dict(best_state); m.eval()
    return m, best_va, ep + 1

configs = [(1e-3, 1e-2, 1e-3), (1e-3, 1e-3, 1e-3), (3e-4, 3e-3, 1e-3)]
trained = []
for cfg in configs:
    m, bv, n_ep = train_q(*cfg)
    trained.append((bv, cfg, m))
    print(f'lr_pre={cfg[0]}, lr_q={cfg[1]}, wd={cfg[2]} -> best val {bv:.4f} ({n_ep} epochs)')

lr_pre=0.001, lr_q=0.01, wd=0.001 -> best val 0.9326 (10 epochs)
lr_pre=0.001, lr_q=0.001, wd=0.001 -> best val 0.9362 (14 epochs)
lr_pre=0.0003, lr_q=0.003, wd=0.001 -> best val 0.9472 (13 epochs)


In [17]:
bv, cfg, best_m = max(trained, key=lambda t: t[0])
with torch.no_grad():
    te = (best_m(Xte_t).argmax(1) == yte_t).float().mean().item()
print('Chosen config:', cfg, '| val', round(bv, 4), '| test acc', round(te, 4))

Chosen config: (0.0003, 0.003, 0.001) | val 0.9472 | test acc 0.9109


In [18]:
class ClassicalHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(2048, 4), nn.Tanh())
        self.mid = nn.Sequential(nn.Linear(4, 4), nn.Tanh())   # quantum circuit ki jagah
        self.out = nn.Linear(4, 2)
    def forward(self, x):
        return self.out(self.mid(self.pre(x)))

def run_model(make_model, seed, lr_pre=3e-4, lr_other=3e-3, wd=1e-3, epochs=40, patience=8):
    torch.manual_seed(seed)
    m = make_model()
    other = [p for n, p in m.named_parameters() if not n.startswith('pre.')]
    opt = torch.optim.Adam([
        {"params": m.pre.parameters(), "lr": lr_pre, "weight_decay": wd},
        {"params": other, "lr": lr_other}])
    best_va, best_state, bad = 0, None, 0
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]
            opt.zero_grad(); crit(m(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            va = (m(Xva_t).argmax(1) == yva_t).float().mean().item()
        if va > best_va:
            best_va, bad = va, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    m.load_state_dict(best_state); m.eval()
    with torch.no_grad():
        te = (m(Xte_t).argmax(1) == yte_t).float().mean().item()
    return best_va, te

seeds = [0, 1, 2, 3, 4, 5]
res = {"quantum": [], "classical": []}
for s in seeds:
    res["quantum"].append(run_model(QuantumNetHead, s))
    res["classical"].append(run_model(ClassicalHead, s))
    print(f'seed {s}: quantum (val,test)={res["quantum"][-1]} | classical={res["classical"][-1]}')

for k in res:
    t = np.array([r[1] for r in res[k]])
    print(f'{k}: test acc mean {t.mean():.4f} +/- {t.std():.4f}')

seed 0: quantum (val,test)=(0.9289617538452148, 0.9018181562423706) | classical=(0.9781420826911926, 0.9509090781211853)
seed 1: quantum (val,test)=(0.9162113070487976, 0.8818181753158569) | classical=(0.9763205647468567, 0.9509090781211853)
seed 2: quantum (val,test)=(0.9599271416664124, 0.9418181777000427) | classical=(0.9781420826911926, 0.9527272582054138)
seed 3: quantum (val,test)=(0.9471766948699951, 0.8963636159896851) | classical=(0.9763205647468567, 0.9545454382896423)
seed 4: quantum (val,test)=(0.9453551769256592, 0.9036363363265991) | classical=(0.9744991064071655, 0.9472727179527283)
seed 5: quantum (val,test)=(0.9125683307647705, 0.8945454359054565) | classical=(0.9781420826911926, 0.9581817984580994)
quantum: test acc mean 0.9033 +/- 0.0186
classical: test acc mean 0.9524 +/- 0.0034


In [19]:
n_layers = 1
res1 = [run_model(QuantumNetHead, s) for s in seeds]
t = np.array([r[1] for r in res1])
print(np.round(t, 4), '| mean', t.mean().round(4), '+/-', t.std().round(4))

[0.8509 0.9091 0.8109 0.8655 0.8909 0.8327] | mean 0.86 +/- 0.0332


In [20]:
n_qubits, n_layers = 4, 6
dev = qml.device("default.qubit", wires=n_qubits)

def entangling_layer():
    qml.CNOT(wires=[0, 1]); qml.CNOT(wires=[2, 3]); qml.CNOT(wires=[1, 2])

@qml.qnode(dev, interface="torch", diff_method="backprop")
def paper_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)               # inputs already x * pi/2
    for l in range(n_layers):
        entangling_layer()
        for i in range(n_qubits):
            qml.RY(weights[l, i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def small_init(t): return nn.init.normal_(t, mean=0.0, std=0.01)

class PaperQuantumHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(2048, 4), nn.Tanh())
        self.q = qml.qnn.TorchLayer(paper_circuit, {"weights": (n_layers, n_qubits)}, init_method=small_init)
        self.out = nn.Linear(4, 2)
    def forward(self, x):
        return self.out(self.q(self.pre(x) * np.pi / 2))

print(PaperQuantumHead()(Xtr_t[:5]).shape)            # expect torch.Size([5, 2])

res_paper = [run_model(PaperQuantumHead, s) for s in seeds]
t = np.array([r[1] for r in res_paper])
print(np.round(t, 4), '| mean', t.mean().round(4), '+/-', t.std().round(4))

torch.Size([5, 2])
[0.9545 0.9436 0.9491 0.9491 0.9545 0.9473] | mean 0.9497 +/- 0.0039
